<a href="https://colab.research.google.com/github/vdkachner-cell/insulin_bioformatics_analysis/blob/main/insulin_pr1oject_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 14.6 MB/s  0:00:00


In [ ]:
import Bio
print(Bio.__version__)

1.88


In [ ]:
from Bio import Entrez, SeqIO

In [ ]:
Entrez.email = "vd@kachnergmail.com"

In [ ]:
handle = Entrez.efetch(db="nucleotide",id="NM_000207", rettype="gb",retmode="text")
record = SeqIO.read(handle, "genbank")
handle.close()

print(record.description)
print("full length:", len(record.seq))

Homo sapiens insulin (INS), transcript variant 1, mRNA
full length: 465


In [ ]:
for feature in record.feature:
  if feature.type == "CDS":
    cds = feature.extract(record.seq)

    print("coding length:" , len(cds))
    print(cds)

AttributeError: 'SeqRecord' object has no attribute 'feature'

In [ ]:
for feature in record.features:
  if feature.type == "CDS":
    cds = feature.extract(record.seq)

    print("coding length:" , len(cds))
    print(cds)


coding length: 333
ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACCCAGCCGCAGCCTTTGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACCTAGTGTGCGGGGAACGAGGCTTCTTCTACACACCCAAGACCCGCCGGGAGGCAGAGGACCTGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCTGGTGCAGGCAGCCTGCAGCCCTTGGCCCTGGAGGGGTCCCTGCAGAAGCGTGGCATTGTGGAACAATGCTGTACCAGCATCTGCTCCCTCTACCAGCTGGAGAACTACTGCAACTAG


In [ ]:
from Bio.SeqUtils import gc_fraction

print("GC content:", round(gc_fraction(cds) * 100, 1), "%")

GC content: 64.6 %


In [ ]:
protein = cds.translate(to_stop=True)

print(protein)
print("Amino acids:", len(protein))

MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN
Amino acids: 110


In [ ]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

analysis = ProteinAnalysis(str(protein))

print("Molecular weight:", round(analysis.molecular_weight(), 1), "Da")
print("Isoelectric point (pI):", round(analysis.isoelectric_point(), 2))

Molecular weight: 11980.8 Da
Isoelectric point (pI): 5.22


In [ ]:
import urllib.request

accessions = {"Human": "P01308", "Pig": "P01315", "Cow": "P01317", "Dog": "P01321"}
seqs = {}

for name, acc in accessions.items():
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
    text = urllib.request.urlopen(url).read().decode()
    lines = text.split("\n")
    print(name, "->", lines[0][:80])
    seqs[name] = "".join(lines[1:])

Human -> >sp|P01308|INS_HUMAN Insulin OS=Homo sapiens OX=9606 GN=INS PE=1 SV=1
Pig -> >sp|P01315|INS_PIG Insulin OS=Sus scrofa OX=9823 GN=INS PE=1 SV=2
Cow -> >sp|P01317|INS_BOVIN Insulin OS=Bos taurus OX=9913 GN=INS PE=1 SV=2
Dog -> >sp|P01321|INS_CANLF Insulin OS=Canis lupus familiaris OX=9615 GN=INS PE=1 SV=1


In [ ]:
from Bio import Align
from Bio.Align import substitution_matrices

aligner = Align.PairwiseAligner()
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5
aligner.mode = "global"

for name in ["Pig", "Cow", "Dog"]:
    aln = aligner.align(seqs["Human"], seqs[name])[0]
    same = sum(1 for a, b in zip(aln[0], aln[1]) if a == b)
    print(name, "identity to human:", round(same / len(seqs["Human"]) * 100, 1), "%")

Pig identity to human: 85.5 %
Cow identity to human: 80.0 %
Dog identity to human: 88.2 %
